# 13 · Трекинг объектов / Object tracking
**RU.** MOT15: реальные уличные видеопоследовательности TUD-Campus (71 кадр, 25 FPS) и TUD-Stadtmitte (179 кадров, 25 FPS). Наблюдение — кадр, вход — RGB/gray и публичные детекции x/y/w/h/confidence, цель — рамка и постоянный ID человека. [Официальный архив](https://motchallenge.net/data/MOT15/). Используйте для учебного исследования с указанием MOTChallenge и авторов TUD, соблюдая условия исходных наборов. Данные не перепубликуются в GitHub.
**EN.** MOT15 contains real street videos: TUD-Campus (71 frames, 25 FPS) and TUD-Stadtmitte (179 frames, 25 FPS). Observations are video frames, inputs are pixels and public x/y/w/h/confidence detections, targets are boxes and persistent pedestrian IDs. Use the official archive for educational research with MOTChallenge/TUD attribution and respect original-source terms. Dataset files are not redistributed here.

**RU.** Первый запуск скачивает 1.31 ГБ, затем использует кэш; CPU, NumPy, SciPy, OpenCV, matplotlib. Никаких случайных кругов вместо людей. TUD-Campus используем для настройки, TUD-Stadtmitte — один раз после фиксации параметров. Обе последовательности принадлежат официальному train: наш test — учебный, не скрытый benchmark. Соседние кадры не перемешиваем. GT разрешён для оценки, но не подаётся трекеру. Наш IoU-трекер не является реализацией Deep SORT.
**EN.** First run downloads 1.31 GB, then reuses the cache. CPU dependencies: NumPy, SciPy, OpenCV, matplotlib. Tune on TUD-Campus and evaluate TUD-Stadtmitte once after freezing parameters. Both are official training sequences; our held-out sequence is a teaching split, not the hidden benchmark. Never shuffle adjacent frames. Use ground truth only for evaluation, not as tracker input. The IoU tracker is not a Deep SORT implementation.


In [ ]:
from pathlib import Path  # RU: задаём пути к кэшу; EN: define cache paths.
import urllib.request, zipfile, hashlib, json  # RU: загружаем и проверяем архив; EN: download and verify the archive.
import numpy as np  # RU: обрабатываем координаты; EN: process coordinates.
import cv2  # RU: вычисляем оптический поток; EN: compute optical flow.
import matplotlib.pyplot as plt  # RU: показываем реальные кадры; EN: display real frames.
from scipy.optimize import linear_sum_assignment  # RU: решаем задачу назначения; EN: solve linear assignment.
data=next((p for p in [Path('build/data'),Path('../build/data'),Path('data')] if (p/'MOT15.zip').exists()),Path('data'))  # RU: находим существующий архив; EN: locate an existing archive.
data.mkdir(parents=True,exist_ok=True)  # RU: создаём кэш при необходимости; EN: create cache if needed.
archive=data/'MOT15.zip'  # RU: задаём имя официального архива; EN: name the official archive.
if not archive.exists():  # RU: скачиваем только отсутствующий файл; EN: download only a missing file.
    urllib.request.urlretrieve('https://motchallenge.net/data/MOT15.zip',archive)  # RU: используем официальный сервер; EN: use the official server.
h=hashlib.sha256()  # RU: создаём потоковую контрольную сумму; EN: initialize streaming checksum.
with archive.open('rb') as f:  # RU: не загружаем весь архив в память; EN: avoid loading the entire archive into memory.
    for chunk in iter(lambda:f.read(1048576),b''): h.update(chunk)  # RU: хешируем блоки по мегабайту; EN: hash one-megabyte chunks.
assert h.hexdigest()=='4197530e8bff30fc9fb55565280b8235ab3cc9d16a2eca3ba95e7d5dcae3a5b4'  # RU: проверяем зафиксированные данные; EN: verify pinned data.
with zipfile.ZipFile(archive) as z:  # RU: открываем архив без произвольной распаковки; EN: open without unrestricted extraction.
    for seq in ['TUD-Campus','TUD-Stadtmitte']:  # RU: выбираем две учебные сцены; EN: select two teaching scenes.
        prefix='MOT15/train/'+seq+'/'  # RU: ограничиваем допустимый путь; EN: restrict the allowed path.
        for name in z.namelist():  # RU: перебираем имена внутри архива; EN: inspect archive members.
            if name.startswith(prefix) and not name.endswith('/') and '..' not in Path(name).parts:  # RU: допускаем только безопасные файлы сцены; EN: allow safe scene files only.
                dest=data/name  # RU: строим локальный путь; EN: construct the local path.
                dest.parent.mkdir(parents=True,exist_ok=True)  # RU: создаём подкаталог; EN: create the subdirectory.
                if not dest.exists(): dest.write_bytes(z.read(name))  # RU: извлекаем файл один раз; EN: extract each file once.
def load_sequence(seq):  # RU: читаем наблюдения и метки отдельно; EN: load observations and labels separately.
    root=data/'MOT15/train'/seq  # RU: выбираем каталог сцены; EN: select the scene directory.
    det=np.loadtxt(root/'det/det.txt',delimiter=',')  # RU: читаем публичные детекции; EN: read public detections.
    gt=np.loadtxt(root/'gt/gt.txt',delimiter=',')  # RU: читаем GT только для оценки; EN: read GT only for evaluation.
    det[:,2:4]-=1;gt[:,2:4]-=1  # RU: переводим x/y из 1-based в 0-based; EN: convert x/y from one-based to zero-based.
    frames=sorted((root/'img1').glob('*.jpg'))  # RU: сохраняем временной порядок; EN: preserve temporal order.
    return det,gt,frames  # RU: возвращаем данные с разными ролями; EN: return data with distinct roles.
det,gt,frames=load_sequence('TUD-Campus')  # RU: начинаем со сцены разработки; EN: start with the development scene.
a=cv2.imread(str(frames[0]),0);b=cv2.imread(str(frames[1]),0)  # RU: читаем соседние реальные кадры; EN: read adjacent real frames.
p0=cv2.goodFeaturesToTrack(a,150,.01,7)  # RU: находим исходные углы; EN: detect initial corners.
p1,s1,_=cv2.calcOpticalFlowPyrLK(a,b,p0,None,winSize=(21,21),maxLevel=3)  # RU: переносим точки вперёд; EN: track points forward.
pback,s2,_=cv2.calcOpticalFlowPyrLK(b,a,p1,None,winSize=(21,21),maxLevel=3)  # RU: возвращаем точки назад; EN: track points backward.


## 1 · Детекции по кадрам / Frame detections
**RU · 5–10 минут.** Реализуйте `frame_boxes(rows,frame,min_score)`: из MOT-массива верните столбцы 2:6 для указанного номера кадра с confidence>=min_score. Сохраните исходный порядок, форму (N,4), dtype float; пустой результат — (0,4). Координаты уже переведены в 0-based. Сравните кадры 1 и 2 при пороге 0, покажите рамки поверх фотографии. GT здесь не используйте.
**EN · 5–10 minutes.** Implement `frame_boxes(rows,frame,min_score)`: select columns 2:6 for the requested frame and confidence>=min_score, preserving row order. Return a float array with shape (N,4), including (0,4) when empty. Coordinates are already zero-based. Compare frames 1 and 2 at threshold 0 and overlay boxes on a photograph. Do not use GT.

In [ ]:
print(det[:2,[0,2,3,4,5,6]])  # RU: изучаем формат детекций; EN: inspect detection format.

In [ ]:
def frame_boxes(rows,frame,min_score):  # RU: фильтруем наблюдения по времени и уверенности; EN: filter observations by time and confidence.
    raise NotImplementedError('TODO / Ваш код')  # RU: реализуйте описанный контракт; EN: implement the stated contract.

In [ ]:
boxes=frame_boxes(det,1,0)  # RU: выбираем первый кадр; EN: select the first frame.
assert boxes.shape[1]==4 and len(boxes)==np.sum((det[:,0]==1)&(det[:,6]>=0))  # RU: сверяем число наблюдений; EN: verify observation count.
assert frame_boxes(det,100000,0).shape==(0,4)  # RU: проверяем пустой кадр; EN: check an empty frame.
fig,ax=plt.subplots();ax.imshow(a,cmap='gray')  # RU: показываем исходное изображение; EN: display the source image.
from matplotlib.patches import Rectangle  # RU: используем рамки для разметки измерений; EN: annotate measured boxes.
for x,y,w,h in boxes: ax.add_patch(Rectangle((x,y),w,h,fill=False,color='lime'))  # RU: накладываем детекции; EN: overlay detections.
plt.show()  # RU: выводим результат; EN: show the result.


## 2 · Проверка потока / Flow validation
**RU · 5–10 минут.** `fb_mask(p0,pback,status_forward,status_backward,threshold)` возвращает bool-вектор длины N. Примите точки, у которых оба статуса равны 1, координаты конечны и евклидова ошибка возврата <=threshold. Входные точки имеют форму (N,1,2). Сравните число точек для 0.5 и 1.0 пикселя. Объясните, почему малый forward-backward error не доказывает правильную идентичность человека.
**EN · 5–10 minutes.** Return a length-N boolean mask accepting points with both status flags equal to one, finite coordinates, and Euclidean return error <=threshold. Points have shape (N,1,2). Compare counts at 0.5 and 1.0 pixels. Explain why a small forward-backward error does not prove pedestrian identity.

In [ ]:
print(np.linalg.norm(pback-p0,axis=2)[:5])  # RU: смотрим ошибки возврата; EN: inspect return errors.

In [ ]:
def fb_mask(p0,pback,status_forward,status_backward,threshold):  # RU: отбираем согласованные треки точек; EN: select consistent point tracks.
    raise NotImplementedError('TODO / Ваш код')  # RU: реализуйте описанный контракт; EN: implement the stated contract.

In [ ]:
keep=fb_mask(p0,pback,s1,s2,1.0)  # RU: применяем порог один пиксель; EN: apply a one-pixel threshold.
expected=(s1.ravel()==1)&(s2.ravel()==1)&np.isfinite(np.linalg.norm(pback-p0,axis=2).ravel())&(np.linalg.norm(pback-p0,axis=2).ravel()<=1)  # RU: независимо проверяем правило; EN: independently verify the rule.
assert keep.dtype==bool and np.array_equal(keep,expected)  # RU: сверяем тип и значения; EN: check type and values.
assert fb_mask(p0,pback,s1,s2,.5).sum()<=keep.sum()  # RU: проверяем монотонность порога; EN: check threshold monotonicity.
print('Accepted:',int(keep.sum()),'of',len(keep))  # RU: выводим измеренное число; EN: report the measured count.


## 3 · IoU и назначение / IoU and assignment
**RU · 5–10 минут.** `iou_matrix(A,B)` получает float-массивы (N,4)/(M,4) в формате x/y/w/h с неотрицательными размерами. Верните (N,M): пересечение / объединение; нулевое объединение даёт 0. Для пустых входов сохраните размерности. Затем изучите готовый `associate`: сначала запрещаются пары ниже gate, затем Hungarian решает назначение, запрещённые пары отбрасываются. Сопоставьте публичные детекции кадров 1 и 2.
**EN · 5–10 minutes.** Implement pairwise IoU for (N,4)/(M,4) x/y/w/h boxes with nonnegative sizes. Return (N,M), using zero for zero union and preserving empty dimensions. Inspect the supplied association routine: gate invalid pairs before Hungarian assignment and discard prohibited assignments afterward. Associate public detections from frames 1 and 2.

In [ ]:
print(det[det[:,0]==1,2:6][:2])  # RU: проверяем формат x/y/w/h; EN: inspect x/y/w/h format.

In [ ]:
def iou_matrix(A,B):  # RU: вычисляем попарное перекрытие рамок; EN: compute pairwise box overlap.
    raise NotImplementedError('TODO / Ваш код')  # RU: реализуйте описанный контракт; EN: implement the stated contract.

In [ ]:
boxes=frame_boxes(det,1,0)  # RU: берём реальные наблюдения; EN: use real observations.
mat=iou_matrix(boxes,boxes)  # RU: сравниваем рамки с собой; EN: compare boxes with themselves.
assert np.allclose(np.diag(mat),1) and np.allclose(mat,mat.T)  # RU: проверяем тождество и симметрию; EN: check identity and symmetry.
assert iou_matrix(np.empty((0,4)),boxes).shape==(0,len(boxes))  # RU: проверяем пустую размерность; EN: verify the empty dimension.


## 4 · Смена идентичности / Identity switches
**RU · 5–10 минут.** `count_switches(events)` получает строки (frame,gt_id,pred_id) после независимого сопоставления с GT. Отсортируйте по frame,gt_id. Для каждого GT сравнивайте текущий pred_id с последним наблюдавшимся, даже после пропуска кадров. Посчитайте изменения. Пустой список даёт 0. Этот договор учебный: не называйте результат официальным IDF1/HOTA. Запустите ниже полный пример, зафиксируйте gate по TUD-Campus и только затем оцените TUD-Stadtmitte. Покажите один фрагмент со сменой ID.
**EN · 5–10 minutes.** Given matched (frame,gt_id,pred_id) events, sort by frame and GT ID. For each GT, compare the predicted ID with its last observed match, including across gaps. Count changes, returning zero for an empty list. This is a teaching convention, not official IDF1/HOTA. Run the complete example below, freeze the gate on TUD-Campus, and then evaluate TUD-Stadtmitte. Inspect a fragment containing an ID change.

In [ ]:
print(gt[:3,:6])  # RU: изучаем эталонные идентификаторы; EN: inspect reference IDs.

In [ ]:
def count_switches(events):  # RU: считаем изменения назначенной идентичности; EN: count assigned-identity changes.
    raise NotImplementedError('TODO / Ваш код')  # RU: реализуйте описанный контракт; EN: implement the stated contract.

In [ ]:
assert count_switches([])==0  # RU: пустая последовательность не содержит переключений; EN: an empty sequence has no switches.
events=[(int(r[0]),int(r[1]),int(r[1])) for r in gt]  # RU: используем реальные GT как идеальный контроль; EN: use real GT as a perfect control.
assert count_switches(events)==0  # RU: постоянный ID не меняется; EN: constant identity must not switch.


## Общий эксперимент / Complete experiment
RU: Использует ваши четыре функции. EN: Uses your four implementations.

In [ ]:
def associate(A,B,gate):  # RU: сопоставляем рамки с ограничением IoU; EN: associate boxes with an IoU gate.
    matrix=iou_matrix(A,B)  # RU: строим матрицу сходства; EN: build the similarity matrix.
    if not matrix.size: return []  # RU: обрабатываем отсутствие наблюдений; EN: handle missing observations.
    rows,cols=linear_sum_assignment(np.where(matrix>=gate,1-matrix,1e6))  # RU: запрещаем плохие пары до назначения; EN: gate poor pairs before assignment.
    return [(int(i),int(j)) for i,j in zip(rows,cols) if matrix[i,j]>=gate]  # RU: исключаем вынужденные запрещённые пары; EN: remove forced prohibited pairs.
def run_tracker(seq,gate):  # RU: выполняем простой IoU-трекинг; EN: run a simple IoU tracker.
    observations,truth,paths=load_sequence(seq)  # RU: разделяем детекции и GT; EN: separate detections and GT.
    active={};next_id=1;prediction=[]  # RU: создаём состояние трекера; EN: initialize tracker state.
    for frame in range(1,len(paths)+1):  # RU: идём по кадрам без перемешивания; EN: process frames without shuffling.
        boxes=frame_boxes(observations,frame,0)  # RU: фильтруем публичные детекции; EN: filter public detections.
        ids=sorted(active)  # RU: обеспечиваем стабильный порядок треков; EN: keep stable track order.
        pairs=associate([active[k][0] for k in ids],boxes,gate)  # RU: связываем прошлые рамки с текущими; EN: link previous and current boxes.
        matched={j for i,j in pairs}  # RU: запоминаем использованные детекции; EN: record used detections.
        for i,j in pairs:  # RU: обновляем существующие треки; EN: update existing tracks.
            active[ids[i]]=(boxes[j],frame)  # RU: сохраняем рамку и последний кадр; EN: store the box and last frame.
            prediction.append([frame,ids[i],*boxes[j]])  # RU: выводим ID только при наблюдении; EN: emit an ID only when observed.
        for j in range(len(boxes)):  # RU: обрабатываем оставшиеся наблюдения; EN: process remaining observations.
            if j not in matched:  # RU: создаём трек для новой детекции; EN: create a track for a new detection.
                active[next_id]=(boxes[j],frame)  # RU: сохраняем новое состояние; EN: store the new state.
                prediction.append([frame,next_id,*boxes[j]]);next_id+=1  # RU: выдаём новый уникальный ID; EN: issue a new unique ID.
        active={k:v for k,v in active.items() if frame-v[1]<=2}  # RU: допускаем два пропущенных кадра; EN: tolerate two missed frames.
    return np.asarray(prediction).reshape(-1,6),truth,len(paths)  # RU: возвращаем выход и данные оценки; EN: return output and evaluation data.
def evaluate_tracks(prediction,truth,nframes):  # RU: считаем учебные метрики; EN: compute teaching metrics.
    fp=fn=tp=0;events=[]  # RU: инициализируем счётчики и историю ID; EN: initialize counts and identity history.
    for frame in range(1,nframes+1):  # RU: оцениваем каждый кадр; EN: evaluate each frame.
        ref=truth[(truth[:,0]==frame)&(truth[:,6]>0)]  # RU: оставляем оцениваемые GT; EN: select evaluated GT boxes.
        out=prediction[prediction[:,0]==frame]  # RU: берём предсказания кадра; EN: select frame predictions.
        pairs=associate(ref[:,2:6],out[:,2:6],.5)  # RU: фиксируем порог оценки отдельно от трекера; EN: fix evaluation gating independently of the tracker.
        tp+=len(pairs);fn+=len(ref)-len(pairs);fp+=len(out)-len(pairs)  # RU: считаем совпадения, пропуски и лишние рамки; EN: count matches, misses and extra boxes.
        events.extend((frame,int(ref[i,1]),int(out[j,1])) for i,j in pairs)  # RU: записываем сопоставленные ID; EN: record matched identities.
    switches=count_switches(events)  # RU: применяем наш договор о переключениях; EN: apply our switch-count convention.
    total=int(np.sum(truth[:,6]>0))  # RU: считаем все оцениваемые GT-рамки; EN: count all evaluated GT boxes.
    return {'tp':tp,'fp':fp,'fn':fn,'id_switches':switches,'mota_like':1-(fp+fn+switches)/total},events  # RU: явно называем метрику учебной; EN: explicitly label the teaching metric.
reports=[]  # RU: собираем только dev-результаты; EN: collect development results only.
for gate in [.1,.3,.5]:  # RU: используем заранее заданные кандидаты; EN: use predeclared candidates.
    pred,truth,nframes=run_tracker('TUD-Campus',gate)  # RU: настраиваем на Campus; EN: tune on Campus.
    metric,_=evaluate_tracks(pred,truth,nframes)  # RU: измеряем качество кандидата; EN: measure candidate performance.
    reports.append({'gate':gate,**metric})  # RU: сохраняем параметры вместе с оценкой; EN: store settings with scores.
chosen=max(reports,key=lambda r:(r['mota_like'],-r['gate']))['gate']  # RU: при равенстве выбираем меньший gate; EN: break ties with smaller gate.
pred,truth,nframes=run_tracker('TUD-Stadtmitte',chosen)  # RU: открываем отдельную сцену после фиксации; EN: evaluate the separate scene after freezing.
metric,events=evaluate_tracks(pred,truth,nframes)  # RU: получаем итог без перенастройки; EN: obtain the final result without retuning.
print('DEV:',reports,'FROZEN:',chosen,'HELD-OUT:',metric)  # RU: показываем воспроизводимый отчёт; EN: display the reproducible report.
assert count_switches(list(reversed(events)))==metric['id_switches']  # RU: проверяем сортировку событий; EN: verify event sorting.
paths=load_sequence('TUD-Stadtmitte')[2]  # RU: получаем реальные тестовые кадры; EN: obtain real test frames.
fig,axes=plt.subplots(1,3,figsize=(15,4))  # RU: создаём последовательный просмотр; EN: create a temporal preview.
for ax,frame in zip(axes,[35,75,120]):  # RU: выбираем фиксированные моменты; EN: select fixed time points.
    ax.imshow(cv2.cvtColor(cv2.imread(str(paths[frame-1])),cv2.COLOR_BGR2RGB))  # RU: показываем исходную фотографию; EN: show the original photograph.
    for row in pred[pred[:,0]==frame]:  # RU: накладываем только предсказания; EN: overlay predictions only.
        x,y,w,h=row[2:6]  # RU: извлекаем геометрию рамки; EN: extract box geometry.
        ax.add_patch(Rectangle((x,y),w,h,fill=False,color='cyan'))  # RU: рисуем измеренную рамку; EN: draw the measured box.
        ax.text(x,y,str(int(row[1])),color='yellow')  # RU: подписываем ID трекера; EN: label the tracker ID.
    ax.set_title('frame '+str(frame));ax.axis('off')  # RU: указываем номер кадра; EN: identify the frame number.
plt.tight_layout();plt.show()  # RU: выводим сравнение; EN: display the comparison.
